# Heart Disease Prediction — SVM
**Dataset**: Cleveland UCI Heart Disease

**Goal**: Binary classification — predict presence of heart disease

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, roc_curve

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Load Dataset

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
cols = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
        'exang','oldpeak','slope','ca','thal','target']
df = pd.read_csv(url, names=cols, na_values='?')
df['target'] = (df['target'] > 0).astype(int)
print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Class balance
df['target'].value_counts().plot(kind='bar', ax=axes[0], color=['#378ADD','#E24B4A'], edgecolor='none')
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['No Disease', 'Disease'], rotation=0)

# Age distribution by target
df.groupby('target')['age'].plot(kind='hist', ax=axes[1], alpha=0.6, bins=20, legend=True)
axes[1].set_title('Age Distribution by Target')
axes[1].legend(['No Disease', 'Disease'])

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Preprocess

In [ ]:
# Fill missing
df['ca'].fillna(df['ca'].median(), inplace=True)
df['thal'].fillna(df['thal'].median(), inplace=True)

# One-hot encode
df = pd.get_dummies(df, columns=['cp','restecg','slope','thal'], drop_first=True)

X = df.drop('target', axis=1)
y = df['target']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('Train:', X_train_sc.shape, '  Test:', X_test_sc.shape)

## 4. Train & Evaluate SVM

In [ ]:
svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm.fit(X_train_sc, y_train)

y_pred = svm.predict(X_test_sc)
y_prob = svm.predict_proba(X_test_sc)[:, 1]

print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['No Disease','Disease'],
    cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#185FA5', lw=2,
             label=f'AUC = {roc_auc_score(y_test, y_prob):.3f}')
axes[1].plot([0,1],[0,1],'--',color='gray')
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#185FA5')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Kernel Comparison

In [ ]:
import sys
sys.path.append('..')
from utils.evaluate import compare_kernels

results = compare_kernels(X_train_sc, y_train, X_test_sc, y_test)

pd.DataFrame(results).T.sort_values('roc_auc', ascending=False)

## 6. GridSearchCV Tuning

In [ ]:
param_grid = {
    'C':     [0.1, 1, 10, 100],
    'kernel':['rbf', 'linear'],
    'gamma': ['scale', 'auto', 0.001, 0.01],
}
grid = GridSearchCV(SVC(probability=True), param_grid,
                    cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train_sc, y_train)

print('Best params :', grid.best_params_)
print('Best CV AUC :', round(grid.best_score_, 3))